# Notas — Aula 4: Python idiomático

Marco: nenhuma versão nova do robô — o robô v6 (Aula 3) já está completo. Hoje ele vira **fonte
de dados**: usamos o log de trajetória para aprender comprehensions (lista/tupla/dict),
`*args`/`**kwargs`, `enumerate`/`zip`, e as três ferramentas funcionais clássicas — `map`,
`filter`, `reduce`.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**, ou
> `Kernel → Restart & Run All`) — a célula seguinte define o `log` usado no notebook inteiro. Se
> aparecer `NameError`, é sinal de que essa célula ainda não foi executada nesta sessão do
> kernel.

In [1]:
# Log de trajetória do robô v6 (enriquecido com comando e status) — EXECUTE esta célula primeiro
log = [
    {"x": 3, "y": 0, "comando": "AVANCAR 3",  "status": "OK"},
    {"x": 3, "y": 0, "comando": "GIRAR ESQ",  "status": "OK"},
    {"x": 3, "y": 2, "comando": "AVANCAR 5",  "status": "PAREDE"},
    {"x": 3, "y": 2, "comando": "GIRAR DIR",  "status": "OK"},
    {"x": 6, "y": 2, "comando": "AVANCAR 4",  "status": "PAREDE"},
    {"x": 6, "y": 2, "comando": "GIRAR ESQ",  "status": "OK"},
    {"x": 6, "y": 4, "comando": "AVANCAR 2",  "status": "OK"},
    {"x": 6, "y": 4, "comando": "GIRAR DIR",  "status": "OK"},
    {"x": 6, "y": 4, "comando": "AVANCAR 3",  "status": "PAREDE"},
]
print(f"{len(log)} passos carregados.")

9 passos carregados.


## List comprehension: recap + filtro

Vocês já viram a forma geral na Aula 2: `[expressão for item in iterável]` — um jeito compacto de
escrever um `for` que constrói uma lista nova. Hoje acrescentamos o filtro: `[expressão for item
in iterável if condição]`. No robô, usamos isso para extrair só os dados que interessam de cada
passo do log, sem escrever um `for` de 3 linhas com `.append()`.

In [2]:
numeros = [1, 2, 3, 4, 5]
pares = [n for n in numeros if n % 2 == 0]
print(pares)

[2, 4]


In [3]:
coords = [(p["x"], p["y"]) for p in log]
print(coords)

passos_ok = [p for p in log if p["status"] == "OK"]
print(len(passos_ok))

[(3, 0), (3, 0), (3, 2), (3, 2), (6, 2), (6, 2), (6, 4), (6, 4), (6, 4)]
6


### Sua vez

Extraia, numa lista, os **comandos** (campo `"comando"`) de todo passo cujo status foi
`"PAREDE"` — os comandos que resultaram em bloqueio.

*Dica: filtro igual ao de `passos_ok` acima, só que testando `"PAREDE"` em vez de `"OK"`, e
extraindo `p["comando"]` em vez do dict inteiro.*

In [4]:
def bloqueios(log):
    return [p["comando"] for p in log if p["status"] == "PAREDE"]


print(bloqueios(log))

['AVANCAR 5', 'AVANCAR 4', 'AVANCAR 3']


## "Tuple comprehension"? Não existe — é generator expression

Intuição natural: colchete faz lista, chave faz dict, será que parêntese faz tupla? **Não.**
`(...)` em volta de `expressão for item in iterável` cria uma **expressão geradora** — um objeto
`generator` que produz valores sob demanda, não uma coleção pronta. Para ter uma tupla de
verdade, é preciso `tuple(...)` por cima. No robô, isso importa quando queremos um registro
**imutável** — por exemplo, uma "foto" fixa de todos os status do log.

In [5]:
g = (n for n in range(3))
print(g)
print(type(g))

tup = tuple(n for n in range(3))
print(tup)

<generator object <genexpr> at 0x103cd92f0>
<class 'generator'>
(0, 1, 2)


In [6]:
status_tup = tuple(p["status"] for p in log)
print(status_tup)

('OK', 'OK', 'PAREDE', 'OK', 'PAREDE', 'OK', 'OK', 'OK', 'PAREDE')


### Sua vez

Crie uma tupla com todos os valores de `x` do log, usando `tuple(...)` sobre uma expressão
geradora (não uma list comprehension).

*Dica: mesma estrutura de `status_tup` acima, trocando `p["status"]` por `p["x"]`.*

In [7]:
def xs_como_tupla(log):
    return tuple(p["x"] for p in log)


print(xs_como_tupla(log))

(3, 3, 3, 3, 6, 6, 6, 6, 6)


## Nested comprehension e a armadilha `[[...]] * N`

Vocês já viram esta armadilha na Aula 2: `[[0] * LADO] * LADO` parece criar uma grade
`LADO`×`LADO`, mas na verdade cria **uma** linha repetida `LADO` vezes — as "linhas" são o mesmo
objeto. Mudar uma célula muda todas. A comprehension resolve porque `for _ in range(LADO)` cria
um objeto novo a cada repetição — `id()` prova a diferença.

In [8]:
LADO = 3
grade_errada = [[0] * LADO] * LADO
grade_errada[0][0] = 9
print(grade_errada)
print(id(grade_errada[0]) == id(grade_errada[1]))

grade_certa = [[0] * LADO for _ in range(LADO)]
grade_certa[0][0] = 9
print(grade_certa)
print(id(grade_certa[0]) == id(grade_certa[1]))

[[9, 0, 0], [9, 0, 0], [9, 0, 0]]
True
[[9, 0, 0], [0, 0, 0], [0, 0, 0]]
False


In [9]:
LADO_GRADE = 4
obstaculos = {(1, 1), (2, 2)}
grade_robo = [[1 if (x, y) in obstaculos else 0 for x in range(LADO_GRADE)]
              for y in range(LADO_GRADE)]
for linha in grade_robo:
    print(linha)

[0, 0, 0, 0]
[0, 1, 0, 0]
[0, 0, 1, 0]
[0, 0, 0, 0]


### Sua vez

Construa uma grade 4×4 marcando com `1` a diagonal principal (onde `x == y`) e `0` no resto,
usando nested comprehension (**sem** `[[...]] * N`).

*Dica: troque a condição `(x, y) in obstaculos` por `x == y`.*

In [10]:
def grade_diagonal(n):
    return [[1 if x == y else 0 for x in range(n)] for y in range(n)]


print(grade_diagonal(4))

[[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]


## Dict comprehension, aprofundado

Mesma ideia, com `{chave: valor for ...}`. Hoje vimos três usos a mais do básico: **inverter** um
dict (trocar chave por valor), **contar sem `Counter`** (dict comprehension sobre um `set`, para
não refazer o trabalho a cada posição repetida), e **achar o máximo sem ordenar tudo**
(`max(dict, key=dict.get)`).

In [11]:
capitais = {"PE": "Recife", "SP": "São Paulo"}
por_cidade = {cidade: uf for uf, cidade in capitais.items()}
print(por_cidade)

{'Recife': 'PE', 'São Paulo': 'SP'}


In [12]:
posicoes = [(p["x"], p["y"]) for p in log]
contagem = {pos: posicoes.count(pos) for pos in set(posicoes)}
print(contagem)

mais_visitada = max(contagem, key=contagem.get)
print(mais_visitada, contagem[mais_visitada])

{(3, 2): 2, (6, 2): 2, (6, 4): 3, (3, 0): 2}
(6, 4) 3


### Sua vez

Crie um dict comprehension que mapeia cada **comando distinto** do log (campo `"comando"`) para
`True`, e devolva quantos comandos distintos existem (`len` do dict).

*Dica: iterar sobre `log` direto (sem `set`) já resolve aqui, porque o valor `True` não muda com
repetição — `{p["comando"]: True for p in log}`.*

In [13]:
def comandos_distintos(log):
    unicos = {p["comando"]: True for p in log}
    return len(unicos)


print(comandos_distintos(log))

6


## `*args`: do exemplo neutro ao robô

Funções que aceitam qualquer número de argumentos — vocês já usaram isso em `print(a, b, c,
...)`. `*args` empacota os argumentos posicionais recebidos numa tupla. No robô, usamos para uma
função de relatório que recebe qualquer quantidade de passos.

In [14]:
def soma_tudo(*numeros):
    return sum(numeros)

print(soma_tudo(1, 2, 3))
print(soma_tudo(10, 20))

valores = [4, 5, 6]
print(soma_tudo(*valores))

6
30
15


In [15]:
def relatorio(*passos):
    return [f"({p['x']}, {p['y']})" for p in passos]

print(relatorio(log[0], log[1], log[2]))
print(relatorio(*log[:3]))

['(3, 0)', '(3, 0)', '(3, 2)']
['(3, 0)', '(3, 0)', '(3, 2)']


### Sua vez

Escreva `maior_valor(*numeros)`, que recebe qualquer quantidade de números e devolve o maior —
**sem** usar a função `max()` embutida.

*Dica: guarde o primeiro número numa variável e compare com cada um dos seguintes.*

In [16]:
def maior_valor(*numeros):
    maior = numeros[0]
    for n in numeros[1:]:
        if n > maior:
            maior = n
    return maior


print(maior_valor(3, 7, 2, 9, 4))

9


## `**kwargs`: do exemplo neutro ao robô

Mesma lógica de `*args`, para argumentos **nomeados**: `**kwargs` empacota tudo num dict. No
robô, usamos para configurar opções sem precisar enumerar cada parâmetro.

In [17]:
def criar_perfil(**dados):
    return dados

print(criar_perfil(nome="Ana", idade=30))

{'nome': 'Ana', 'idade': 30}


In [18]:
def configurar_robo(**opcoes):
    lado = opcoes.get("lado", 10)
    inicio = opcoes.get("inicio", (0, 0))
    return lado, inicio

print(configurar_robo(lado=8, inicio=(2, 2)))
print(configurar_robo())

(8, (2, 2))
(10, (0, 0))


### Sua vez

Escreva `resumo(**dados)` que imprime cada par recebido como `"chave: valor"`, um por linha.

*Dica: `dados` é um dict dentro da função — use `.items()` para percorrer os pares.*

In [19]:
def resumo(**dados):
    for chave, valor in dados.items():
        print(f"{chave}: {valor}")


resumo(x=3, y=0, direcao="LESTE")

x: 3
y: 0
direcao: LESTE


## `enumerate` e `zip`

`enumerate` numera um iterável sem precisar de um contador manual. `zip` combina duas ou mais
sequências, posição a posição — inclusive dá para transformar duas listas paralelas num dict
com `dict(zip(chaves, valores))`.

In [20]:
nomes = ["Ana", "Bruno"]
notas = [8.5, 6.0]
print(list(zip(nomes, notas)))
print(dict(zip(nomes, notas)))

[('Ana', 8.5), ('Bruno', 6.0)]
{'Ana': 8.5, 'Bruno': 6.0}


In [21]:
for i, passo in enumerate(log[:3], start=1):
    print(f"Passo {i}: {passo['status']}")

comandos_enviados = [p["comando"] for p in log]
resultados = [p["status"] for p in log]
for i, (cmd, res) in enumerate(zip(comandos_enviados, resultados), start=1):
    print(f"{i:2}. {cmd:12} → {res}")

Passo 1: OK
Passo 2: OK
Passo 3: PAREDE
 1. AVANCAR 3    → OK
 2. GIRAR ESQ    → OK
 3. AVANCAR 5    → PAREDE
 4. GIRAR DIR    → OK
 5. AVANCAR 4    → PAREDE
 6. GIRAR ESQ    → OK
 7. AVANCAR 2    → OK
 8. GIRAR DIR    → OK
 9. AVANCAR 3    → PAREDE


### Sua vez

Dadas duas listas paralelas `produtos` e `precos`, escreva `linhas_produtos(produtos, precos)`
que devolve uma lista de strings `"produto: R$preco"` (preço com 2 casas decimais), usando `zip`
+ comprehension.

*Dica: `f"{produto}: R${preco:.2f}"` formata o preço com 2 casas.*

In [22]:
def linhas_produtos(produtos, precos):
    return [f"{produto}: R${preco:.2f}" for produto, preco in zip(produtos, precos)]


print(linhas_produtos(["parafuso", "porca", "arruela"], [0.5, 0.2, 0.1]))

['parafuso: R$0.50', 'porca: R$0.20', 'arruela: R$0.10']


## `map`, e a primeira `lambda`

`map(funcao, iteravel)` aplica uma função a cada elemento, devolvendo um objeto `map` (preguiçoso,
como o generator — precisa de `list(...)` para ver o conteúdo). `lambda parametro: expressão` é
uma função sem nome, útil quando a transformação é curta e só vai ser usada uma vez. Comprehension
geralmente é preferida a `map`+`lambda` em Python — mas reconhecer os dois é importante.

In [23]:
numeros = [1, 2, 3, 4]
print(list(map(str, numeros)))

dobrados = list(map(lambda n: n * 2, numeros))
print(dobrados)

['1', '2', '3', '4']
[2, 4, 6, 8]


In [24]:
acoes = list(map(lambda p: p["comando"].split()[0], log))
print(acoes)

['AVANCAR', 'GIRAR', 'AVANCAR', 'GIRAR', 'AVANCAR', 'GIRAR', 'AVANCAR', 'GIRAR', 'AVANCAR']


### Sua vez

Escreva `quadrados(numeros)` que usa `map` + `lambda` para elevar cada número da lista ao
quadrado.

In [25]:
def quadrados(numeros):
    return list(map(lambda n: n ** 2, numeros))


print(quadrados([1, 2, 3, 4]))

[1, 4, 9, 16]


## `filter`

`filter(funcao, iteravel)` mantém só os elementos para os quais a função devolve `True` — o
parceiro de `map`. De novo, comprehension com `if` geralmente é a forma mais lida; `filter`
aparece bastante em código de quem vem de outras linguagens funcionais.

In [26]:
numeros = [1, 2, 3, 4]
pares = list(filter(lambda n: n % 2 == 0, numeros))
print(pares)

[2, 4]


In [27]:
passos_ok = list(filter(lambda p: p["status"] == "OK", log))
print(len(passos_ok))

6


### Sua vez

Escreva `maiores_que(numeros, limite)` que usa `filter` + `lambda` para manter só os números
maiores que `limite`.

In [28]:
def maiores_que(numeros, limite):
    return list(filter(lambda n: n > limite, numeros))


print(maiores_que([1, 2, 3, 4], 2))

[3, 4]


## `reduce`

`map` transforma, `filter` seleciona, `reduce` **combina** todos os elementos num valor só —
diferente dos outros dois, não é builtin (`from functools import reduce`) e não tem equivalente
direto em comprehension. Prefira um builtin (`sum`, `max`, `min`) quando ele existir; `reduce` é
para quando não existe atalho pronto.

In [29]:
from functools import reduce

numeros = [1, 2, 3, 4]
soma = reduce(lambda acc, n: acc + n, numeros)
print(soma, sum(numeros))

produto = reduce(lambda acc, n: acc * n, numeros)
print(produto)

10 10
24


In [30]:
valores_avancar = [int(p["comando"].split()[1]) for p in log
                   if p["comando"].startswith("AVANCAR")]
total = reduce(lambda acc, v: acc + v, valores_avancar)
print(total, sum(valores_avancar))

17 17


### Sua vez

Escreva `concatenar_comandos(log)` que usa `reduce` para concatenar todos os `"comando"` do log
numa única string, separados por `" | "`.

*Dica: extraia a lista de comandos primeiro (list comprehension), depois aplique `reduce`.*

In [31]:
def concatenar_comandos(log):
    comandos = [p["comando"] for p in log]
    return reduce(lambda acc, c: acc + " | " + c, comandos)


print(concatenar_comandos(log))

AVANCAR 3 | GIRAR ESQ | AVANCAR 5 | GIRAR DIR | AVANCAR 4 | GIRAR ESQ | AVANCAR 2 | GIRAR DIR | AVANCAR 3


## Para aprofundar

- List/tuple/dict comprehensions e generator expressions — Tutorial oficial:
  https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions
- `*args`/`**kwargs` — Tutorial oficial: https://docs.python.org/3/tutorial/controlflow.html#arbitrary-argument-lists
- `enumerate`: https://docs.python.org/3/library/functions.html#enumerate ·
  `zip`: https://docs.python.org/3/library/functions.html#zip
- `lambda`: https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions
- `map`/`filter`: https://docs.python.org/3/library/functions.html#map ·
  https://docs.python.org/3/library/functions.html#filter
- `functools.reduce`: https://docs.python.org/3/library/functools.html#functools.reduce